In [ ]:
using Pkg
Pkg.activate("./")

In [ ]:
using Makie
using CairoMakie
using Dierckx
using HDF5
using LinearAlgebra
using Rotations
using ProgressMeter

# Load up the base triangulation and rotate to new frame
## First load up the vertices of the previous triangulation

In [ ]:
location = "./triangulation.h5:base_triangulation"
filename, groupname = split(location, ":")

original_vertices = h5open(filename) do h5f
    group = h5f[groupname]
    deg2rad.(group["vertices"][:, :])
end

## Define some geometry helpers for rotating

In [ ]:
function longlat_to_cart(long, lat)
    return [cos(long) * cos(lat), sin(long) * cos(lat), sin(lat)]
end

function cart_to_longlat(x, y, z)
    n = norm([x, y, z])
    x, y, z = [x, y, z] ./ n
    return [atan(y, x), asin(z)]
end

longlat0 = deg2rad(-72.279397), deg2rad(-15.622267)

axis = cross([0, 0, 1], longlat_to_cart(longlat0...))
angle = acos(dot([0, 0, 1], longlat_to_cart(longlat0...)))
rotation = AngleAxis(angle, axis...)

## Rotate all the points to be centered around the new point of interest

In [ ]:
rotated_points = [rotation * longlat_to_cart(original_vertices[idx, :]...) for idx in 1:size(original_vertices)[1]]
rotated_longlats = [cart_to_longlat(x...) for x in rotated_points]

vertices = zeros(size(original_vertices))
for idx in 1:length(rotated_longlats)
    vertices[idx, :] = rotated_longlats[idx]
end

fig = Figure()
ax = Axis(
    fig[1, 1],
    xlabel="Longitude [deg.]",
    ylabel="Latitude [deg.]"
)

scatter!(
    ax,
    rad2deg.(vertices[:, 1]),
    rad2deg.(vertices[:, 2]),
    alpha=0.02,
    markersize=3
)

fig

In [ ]:
# rearth = 6_378_000
# pts = zeros((size(vertices, 1), 3))
# for (idx, vertex) in enumerate(eachrow(vertices))
#     if ~(minlat < vertex[2] < maxlat)
#         continue
#     elseif ~(minlong < vertex[1] < maxlong)
#         continue
#     end
#     elev = itp(vertex...)
#     pt = longlat_to_cart(vertex...) * (rearth + elev)
#     pts[idx, :] = pt
# end

# Make topography from previously computed triangulation

This assumes that you have run the other notebook first and have downloaded `earth_relief_01m.grd` from `http://oceania.generic-mapping-tools.org/` and have placed it in this directory. We're also going to write this to an `HDF5` file so that we can access it bit by bit, without readin the whole thing into memory, which is a pain.

## Read the `.grd` file and save to `HDF5`
I've commented this block for now because it can take a very long time to run and should only be done once.

In [ ]:
# using GMT: gmtread
# g = gmtread("./earth_relief_15s.grd");
# h5open("earth_relief_15s.h5", "w") do h5f
#     h5f["longitude"] = Float32.(g.x)
#     h5f["latitude"] = Float32.(g.y)
#     h5f["elevation"] = g.z
# end

## Load the longitude and latitude points

In [ ]:
longs, lats = h5open("earth_relief_15s.h5") do h5f
    longs, lats = deg2rad.(h5f["longitude"][1:end-1]), deg2rad.(h5f["latitude"][1:end-1])
end

## Compute the spline for subsamples of the Earth
The elevation map has too many points to spline all at once so we have to split it up into subsections.
We have chosen to do do this in slices of longitude for simplicity.
Below I have some helper functions for this task.

In [ ]:
function find_indexes(
    long1::Real,
    long2::Real,
    longs::Vector,
    padding::Real,
    branch::Tuple{Real, Real}=(-π, π)
)::Tuple{Vector{Int}, Vector{Float64}}
    @assert all(diff(longs) .> 0) "Longitudes not sorted"
    @assert branch[2] - branch[1]==2π "Branch not full circle"
    @assert (branch[1] <= long1 <= branch[2]) && (branch[1] <= long2 <= branch[2])
    longmin, longmax = long1 - padding, long2 + padding
    long_idxs, spl_longs = Int[], Float32[]
    if longmin < branch[1]
        idxs = findall(2π + longmin .< longs)
        long_idxs = vcat(long_idxs, idxs)
        spl_longs = vcat(spl_longs, longs[idxs] .- 2π)
    end
    idxs = findall(longmin .<= longs .< longmax)
    if length(idxs) > 0
        long_idxs = vcat(long_idxs, idxs)
        spl_longs = vcat(spl_longs, longs[idxs])
    end
    if branch[2] < longmax
        idxs = findall(longs .< longmax - 2π)
        long_idxs = vcat(long_idxs, idxs)
        spl_longs = vcat(spl_longs, longs[idxs] .+ 2π)
    end
    return long_idxs, spl_longs
end

In [ ]:
function split_to_contiguous_ranges(idxs::Vector{Int})::Vector{AbstractUnitRange}
    ranges = UnitRange[]    
    l = first(idxs)
    for (cur, next) in zip(idxs, idxs[2:end])
        if next==cur+1
            continue
        end
        push!(ranges, l:cur)
        l = next
    end
    push!(ranges, l:last(idxs))
    return ranges
end

In [ ]:
degree = 2
step = deg2rad(5)
padding = step / 20
we = -π:step:π

vertex_elevations = zeros(size(vertices, 1))

@showprogress for (long1, long2) in zip(we, we[2:end])
    
    long_idxs, spl_longs = find_indexes(long1, long2, longs, padding)
    ranges = split_to_contiguous_ranges(long_idxs)
    
    elevs = h5open("./earth_relief_15s.h5") do h5f
        elevs = nothing
        for range in ranges
            if elevs==nothing
                elevs = h5f["elevation"][1:length(lats), range]
            else
                elevs = hcat(elevs, h5f["elevation"][1:length(lats), range])
            end
        end
        elevs
    end
#     println(size(elevs))
    itp = Spline2D(spl_longs, lats, elevs'; kx=degree, ky=degree, s=0.0)
    
    idxs = long1 .< vertices[:, 1] .<= long2
    
    vertex_elevations[idxs] = map(x->maximum([itp(x...), 1]), eachrow(vertices)[idxs])

end

vertex_elevations

In [ ]:
# Δlong = 0.01
# Δlat = 0.01
# longs = deg2rad.(g.x)
# lats = deg2rad.(g.y)

# minlong, maxlong = longlat0[1] - Δlong, longlat0[1] + Δlat
# minlat, maxlat = longlat0[2] - Δlat, longlat0[2] + Δlat

# mlong = minlong .< longs[1:end-1] .< maxlong
# mlat = minlat .< lats[1:end-1] .< maxlat

# itp = Spline2D(longs[1:end-1][mlong], lats[1:end-1][mlat], transpose(g.z)[mlong, mlat]; kx=3, ky=3, s=0.0)

# Compute the elevation for each point
The datafile we're using has the depths of the oceans at every point(!!?), but we are going to wrap the PREM in a shell and so we are going to ignore all values below sea level.
I also introduce a hack to make sure the shell is one centimeter off the PREM to avoid issues when ray tracing.

In [ ]:
# elevations = []
# maxx = maximum(x)
# maxy = maximum(y)

# for vertex in eachrow(vertices)
#     x = minimum([vertex[1], maxx])
#     y = minimum([vertex[2], maxy])
#     push!(elevations, maximum([itp(x, y), 1.0]))
# end

In [ ]:
fig = Figure()
ax = Axis(fig[1, 1])

sc = scatter!(
    ax,
    rad2deg.(vertices[:, 1]),
    sin.(vertices[:, 2]),
    color=vertex_elevations,
    alpha=0.01
)

cbar = Colorbar(fig[1, 2], sc, label="Elevation [m]")

fig

# Convert these to trinagles in 3D space

In [ ]:
faces = h5open(filename) do h5f
    group = h5f[groupname]
    faces = group["faces"][:, :]
end;

In [ ]:
vertices

In [ ]:
rearth = 6_378_000 # m

vertices_3d = zeros((size(vertices, 1), 3))

MAGIC_NUMBER = 1

for (idx, elevation) in enumerate(vertex_elevations)
    vertices_3d[idx, :] = longlat_to_cart(vertices[idx, :]...) * (rearth + MAGIC_NUMBER * elevation)
end

In [ ]:
h5open(filename, "r+") do h5f
    if "colca_valley" in keys(h5f)
        delete_object(h5f["colca_valley"])
    end
    group = create_group(h5f, "colca_valley")
    attrs(group)["longitude"] = longlat0[1]
    attrs(group)["latitude"] = longlat0[2]
    attrs(group)["unit"] = "m"
    attrs(group)["rearth"] = rearth
    group["faces"] = faces
    group["vertices"] = vertices_3d
end

In [ ]:
colors = []
for vertex in eachrow(vertices_3d)
    push!(colors, norm(vertex) - rearth)
end
minimum(colors), maximum(colors)

In [ ]:
minimum(colors[colors.> 51])

In [ ]:
fig = Figure(size = (600, 600))
ax = Axis3(fig[1,1], azimuth=deg2rad(-70))
hidedecorations!(ax)
hidespines!(ax)

mesh!(
    ax,
    vertices_3d,
    faces,
    color=colors,
    colormap=Reverse(:speed),
    colorrange=(50.1, 4000*MAGIC_NUMBER),
    lowclip=:navy
)

save("whole_world.png", fig, dpi=1000)

display(fig)

# This can be extended to any other elevation profile you want
Let's try to make an idelaized valley.
Note that these numbers are enormous, and are most so that we can see anything in the plot.

In [ ]:
incline = deg2rad(35)
h1 = 2_000
h2 = 500_000
l = (h2 - h1) / tan(incline)

function perfect_valley_fxn(long, lat)
    Δx = abs(lat-longlat0[2])
    return minimum([(h2-h1) * Δx / tan(incline) + 2000, h2])
end

In [ ]:
rearth = 6_378_000 # m

vertices_3d = Point3f[]
for vertex in eachrow(vertices)
    elevation = perfect_valley_fxn(vertex...)
    push!(vertices_3d, Point3f(longlat_to_cart(vertex...) * (rearth + elevation)))
end

In [ ]:
fig = Figure(size = (600, 600))
ax = Axis3(fig[1,1], azimuth=deg2rad(-70))
mesh!(ax, vertices_3d, faces)

display(fig)

I guess it's more of a Great Valley, and now it runs west-east instead of north-south, but hopefully we will manage.
It also might make sense to make the valley taper off with a sigmoid or something, but we can experiment with that as it becomes an issue